# Inspect MOJAVE preprocessing

Measurements and interpolation use `radio_diffusion.data.preprocess_mojave` on the full, native-beam CLEAN images before cropping. Selected images and ridge points remain in memory for inspection; no processed files are written.

UV matches are listed alongside the image inputs. The current measurements use the image FITS data and its CLEAN-component table.

In [ ]:
import random

from radio_diffusion.paths import get_path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.visualization import AsinhStretch, ImageNormalize
from astropy.wcs import WCS
from IPython.display import display
from matplotlib.transforms import Affine2D

from radio_diffusion.data.preprocess_mojave import (
    JET_ORIENTATION_SETTINGS,
    NOISE_METHOD,
    RIDGELINE_SETTINGS,
    get_noise_jy_per_beam,
    interpolate_ridgeline,
    iter_observations,
    jet_orientation_deg,
    peak_brightness_jy_per_beam,
    total_flux_jy,
    trace_jet_ridgeline,
)

image_dir = get_path("MOJAVE_FITS_DIR")
uv_dir = get_path("MOJAVE_UV_DIR")

n_examples = 10
random_seed = 42
selected_filenames = []  # Empty selects a reproducible random sample.

noise_method = "four_corner"
orientation_settings = JET_ORIENTATION_SETTINGS.copy()
orientation_settings = {
    "inner_mas": 1.0,
    "outer_mas": 3.0,
    "sigma_cut": 3.0,
}
ridgeline_settings = RIDGELINE_SETTINGS.copy()

interpolation_step_mas = 0.05
view_radius_mas = None  # None displays the full image; a number sets the plot radius.

In [ ]:
if not image_dir.is_dir():
    raise FileNotFoundError(f"Image directory not found: {image_dir}")
if not isinstance(n_examples, int) or n_examples < 1:
    raise ValueError("n_examples must be a positive integer.")

observations = list(iter_observations(str(image_dir), str(uv_dir)))
if not observations:
    raise FileNotFoundError(f"No MOJAVE image FITS files found in {image_dir}")

if selected_filenames:
    observations_by_name = {image.name: (image, uv) for image, uv in observations}
    missing = [name for name in selected_filenames if name not in observations_by_name]
    if missing:
        raise FileNotFoundError(f"Requested images not found: {missing}")
    selected_observations = [observations_by_name[name] for name in selected_filenames]
else:
    selected_observations = random.Random(random_seed).sample(
        observations, min(n_examples, len(observations)),
    )

print(f"Images found: {len(observations)}; selected: {len(selected_observations)}")
display(pd.DataFrame([
    {
        "image_filename": image.name,
        "uv_filename": uv.name if uv is not None else None,
    }
    for image, uv in selected_observations
]))

In [ ]:
inspection_examples = []
measurement_rows = []

for image_path, uv_path in selected_observations:
    row = {"image_filename": image_path.name, "error": ""}
    try:
        with fits.open(image_path, memmap=False) as hdul:
            image_hdu = hdul[0]
            clean_components_hdu = hdul["AIPS CC"]
            noise = get_noise_jy_per_beam(image_hdu, noise_method)
            jet_pa = jet_orientation_deg(
                image_hdu, clean_components_hdu, noise, **orientation_settings,
            )
            ridge, stopping_reason = trace_jet_ridgeline(
                image_hdu, noise, initial_pa_deg=jet_pa, **ridgeline_settings,
            )
            metadata = {
                "source_name": image_path.name.split(".", 1)[0],
                "noise_rms_jy_per_beam": noise,
                "noise_method": noise_method,
                "total_flux_jy": total_flux_jy(clean_components_hdu),
                "peak_brightness_jy_per_beam": peak_brightness_jy_per_beam(image_hdu),
                "jet_pa_deg": jet_pa,
                "jet_radial_extent_mas": ridge[-1]["radius_mas"] if ridge else float("nan"),
                "ridgeline_stop_reason": stopping_reason,
                "ridgeline_arc_clipped": any(point["arc_clipped"] for point in ridge),
            }
            image = np.squeeze(image_hdu.data).copy()
            header = image_hdu.header.copy()
        row.update(metadata)
        inspection_examples.append({
            "image_filename": image_path.name,
            "image": image,
            "header": header,
            "metadata": metadata,
            "ridge": ridge,
            "ridgeline_settings": ridgeline_settings.copy(),
        })
    except Exception as error:
        row["error"] = f"{type(error).__name__}: {error}"
    measurement_rows.append(row)

measurement_table = pd.DataFrame(measurement_rows)
with pd.option_context("display.max_colwidth", None):
    display(measurement_table)
print(f"Ready for plotting: {len(inspection_examples)} / {len(selected_observations)} observations")

In [ ]:
def show_example(example, view_radius_mas=None):
    """Display the native image, shared measurements and temporary ridge points."""
    image = example["image"]
    header = example["header"]
    metadata = example["metadata"]
    ridge = example["ridge"]
    height, width = image.shape
    core_pixel = np.array([(width - 1) / 2, (height - 1) / 2])

    # Local east/north offsets from the celestial WCS linear scale matrix.
    scale = WCS(header).celestial.pixel_scale_matrix * 3_600_000
    matrix = np.eye(3)
    matrix[:2, :2] = scale
    matrix[:2, 2] = -scale @ core_pixel
    pixel_to_mas = Affine2D(matrix)
    corners = np.array([[0, 0], [width - 1, 0], [0, height - 1], [width - 1, height - 1]])
    corner_offsets = (corners - core_pixel) @ scale.T
    if view_radius_mas is not None and (
        not np.isfinite(view_radius_mas) or view_radius_mas <= 0
    ):
        raise ValueError("Plot radius must be finite and positive, in mas.")

    peak = metadata["peak_brightness_jy_per_beam"]
    norm = ImageNormalize(
        vmin=0, vmax=max(peak, np.finfo(float).tiny),
        stretch=AsinhStretch(0.01), clip=True,
    )
    fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
    artist = ax.imshow(
        image, origin="lower", cmap="inferno", norm=norm,
        interpolation="nearest", transform=pixel_to_mas + ax.transData,
    )
    detection_level = (
        example["ridgeline_settings"]["sigma_cut"] * metadata["noise_rms_jy_per_beam"]
    )
    if np.nanmin(image) < detection_level < peak:
        ax.contour(
            image, levels=[detection_level], colors="white", linewidths=0.6,
            linestyles=":", transform=pixel_to_mas + ax.transData,
        )
        ax.plot([], [], ":", color="white", label="Ridge detection threshold")

    ax.plot(0, 0, "+", color="white", markersize=10, label="Assumed core")
    direction = np.deg2rad(metadata["jet_pa_deg"])
    ax.annotate(
        "", xy=(3*np.sin(direction), 3*np.cos(direction)), xytext=(0, 0),
        arrowprops={"arrowstyle": "->", "color": "cyan", "lw": 1.8},
    )
    ax.plot([], [], color="cyan", label="Inner-jet direction (1 mas arrow)")

    if ridge:
        raw_pixels = np.array([[point["x_pixel"], point["y_pixel"]] for point in ridge])
        raw_offsets = (raw_pixels - core_pixel) @ scale.T
        ax.plot(
            raw_offsets[:, 0], raw_offsets[:, 1], ".", color="magenta",
            markersize=4, label="Raw ridge detections",
        )
        curve = interpolate_ridgeline(ridge, radial_step_mas=interpolation_step_mas)
        curve_pixels = np.array([[point["x_pixel"], point["y_pixel"]] for point in curve])
        curve_offsets = (curve_pixels - core_pixel) @ scale.T
        ax.plot(
            curve_offsets[:, 0], curve_offsets[:, 1], color="deepskyblue",
            linewidth=1.2, label="Interpolated ridge",
        )
        east, north = raw_offsets[-1]
        ax.plot(
            [0, east], [0, north], "o--", color="lime", markersize=4,
            label=f"Radial extent: {metadata['jet_radial_extent_mas']:.2f} mas",
        )

    noise_text = (
        f"Noise ({metadata['noise_method']}): "
        f"{metadata['noise_rms_jy_per_beam']:.3g} Jy/beam"
    )
    status_text = f"Stop: {metadata['ridgeline_stop_reason']}"
    if metadata["ridgeline_arc_clipped"]:
        status_text += "; detected arc window clipped"
    ax.set_title(
        f"{example['image_filename']}\n"
        f"PA: {metadata['jet_pa_deg']:.2f} deg | {noise_text}\n{status_text}",
        fontsize=10,
    )
    if view_radius_mas is None:
        ax.set_xlim(corner_offsets[:, 0].max(), corner_offsets[:, 0].min())
        ax.set_ylim(corner_offsets[:, 1].min(), corner_offsets[:, 1].max())
    else:
        ax.set_xlim(view_radius_mas, -view_radius_mas)
        ax.set_ylim(-view_radius_mas, view_radius_mas)
    ax.set(xlabel="East offset (mas)", ylabel="North offset (mas)", aspect="equal")
    ax.set_facecolor("black")
    ax.legend(loc="upper right", fontsize=7, facecolor="black", labelcolor="white")
    fig.colorbar(artist, ax=ax, label="Jy/beam", fraction=0.046, pad=0.04)
    plt.show()
    plt.close(fig)

In [ ]:
for example in inspection_examples:
    show_example(example, view_radius_mas=view_radius_mas)